## Staffing overview 

There are a LOT of errors in the staffing data. In many cases, they change by a factor of 10  periodically for no reason . It affects more than 10 percent of the reports. 

Sometimes the number of staff seems to be off by a factor of 10 -- either 100th of the real value, or 100x the real value. But sometimes the number is just crazy off for no apparent reason. (Example, total staff will quietly sit at 20 for all the other years, but be 7 for one year)

The errors more often hit the larger facilties, particularly government ones. 

This is an attempt to fix them where I can and remove them where I can't. 

One difference from others is that I'll also look at total staff, not direct just direct care. The reason is I can see at times that this isn't right -- they'll have nursing staff and not direct care, for example. But it HAS to be higher. A credibly minimim of a 6-person place where they are there all the time is 21 shifts of direct care and 7 shifts of custodial, and possibly 1/2 position of adminsitrative. That comes out to just about 7 people per place.  (5.25 direct care, 1.75 other)


The key values: 

* **Direct care staff** vs **Total staff** : The rules and the important values are direct care staff, so we'll use them when we can. However, these can fluctuate a lot, and seem to be in error more than the total staff, so we will check on how much bigger total staff is. 

* **Licensed beds** vs. **residents** : In the vast majority of places, this doesn't matter that much -- for the 6-person, the minimum is larger than the 

The process: 

1. Drop facilities without enough years' of operation. I'm dropping anything that wasn't still in business in 2024, and also those that have less than 5 years' operation. This will get rid of the newest ones - there are very few. 

2. Adjust the rules from [this paper from Stanford](...) . Here are their rules: 

    1. There must be at least 1 direct care staff on duty all the time. They estimated 80 hours, or 2 ftes for direct care. 
    2. The least stringent ratio is 1 direct care per 6.4 residents, so they estimated less than .3125 FTE is impossible. 
    3. A credible ceiling for a facility that only houses the most profoundly disabled children is 1.325 per resident. 

    Those are direct care staff. The direct care number is odd, so we might need to check against the total staff instead. That means our high number would be considerably higher than the cutoff. 

The cutoffs I'm using to check / remove are: 

* Less than 3 total staff or 2 direct care 
* Less than .25 FTE direct care FTE total staff. This is wildly low but we know some places break the rules. 

The number of staff seems to be exactly 1/10th, 1/100th, or 100x the truth. I have no idea how this happens so often -- It's not one company, inspector, state, etc. There may be implied  decimal places in the data entry? But it's just weird.

The Stanford paper took out surveys that had implausible or illegal values. I'm not sure we want to do that, but here are their rules:

1. Must be a direct care staff member whenever there are clients. Estimated at least 80 hours where there must be clients not in day programs, so minimum of 2.
2. Law says the the least stringent ratio is 1 per 6.4 residents. Less than .3125 FtE per resident was dropped 
3. A ceiling would be credible, takes the most stringent ratio (all there all time, all are children, at highest level, etc.) would be 1.3125 per resident. She multiplied by 3 to take account of places that voluntarily overstaffed. (Note to self: This failed in places that have falling resident counts.)
4. Dropped surveys in which the number in the middle was way off from the median for the facility. (>2 or <.5 from its neighbors, and the median for the facility ), checked against a possible spike or drop in the number of residents.

I think we'd be less stringent:
*   at least 1 staff member, direct carea maybe less than .2, and way higher on ceiling, say 5* the max.
* Then look for those that are 1/10th .


Generally, there are some facilities -- particularly some of the super big ones in Texas and a few other places -- where the staff numbers just look random. It's really hard to pick out what it should be -- it's not an obvious 10-fold difference. In general, we'll flag really, really bad outliers after the adjustments are made conservatively, then just NA those. For our purposes, we care about the ones in NC, and we care about the general trends, particularly in large facilities. But we are OK with having quite a few missing values if they will screw up comparisons. We're not trying to publish averages for each facility -- just broad trends.

I plan not to use any facility for comparisons that: 

* Went out of business before 2020
* Didn't have at least 5 years' of operation by 2026





In [ ]:
pacman::p_load(
  "tidyverse",
  "lubridate",
  "janitor",
  "googledrive",
  "readxl",
  "here"
)


suppressMessages(here::i_am("cms_3070g/01-read-cms-3070g.ipynb" ))
# I'm never sure about paths - just making sure here.
setwd(here())



provider_dates <- readRDS("qcor/data/qcor_provider_master.RDS") |> 
  select ( facility_id = ccn, certified_beds, begin_date, termination_date, years_included) |> 
  mutate ( use_for_comparison = 
                  case_when ( years_included >= 5 & is.na(termination_date) ~ TRUE,
                              years_included >= 5 & termination_date >= "2020-01-01" ~ TRUE,
                              .default = FALSE
                          )
          )    
          

staff_df <-  readRDS("cms_3070g/data/cms3070g_allfields.RDS") |> 
  select ( facility_id = provider_ccn, facility = name_of_facility, state=state_code,  survey_date,
          icfiid_beds, residents = resident_estimate,  dc_staff = direct_care_personnel, staff= total_personnel, staff_ratio)  |> 
  semi_join ( provider_dates |>  filter (use_for_comparison == TRUE) , join_by ( facility_id) )
  







## Check the extent of the problem

* About `r count()

#### Helper function to compute the medians

This creates a median function that gets rid of the current value as part of that median. In other words, it is the median of every valid value EXCEPT this one within a provider. (This could be done in map_dbl() phrase inside a mutate statement but it confused me so I wanted to break it out in a loop instead.)

#### Set up medians

Create a dataset that has the median values based on some very broad valid ranges (not realistic in many areas, but can't be outside these very broad areas)

Each staff number is accompanied by a median for its facility that leaves out its own value and any value outside the range. If they're all  no good, then the whole thing gets an NA for the fixed staff . 

In [31]:
# this works on staff . The minimum valid staff level is 1 as a default. 
# This is just a very basic screen - there are also problems at the  mx, and fixing the under 1 is also a problem. but it's  a screen for the median not the value.

get_neighbors <- function (x, min_valid_staff=1, max_valid_staff = 1500)  {
      #create a list of NA values of the list of x
      result <- rep(NA_real_, length (x) )

      for (i in seq_along (x) ) {
          #remove this value
          all_others <- x[-i]
          # check for minimally valid values
          all_others <- all_others[dplyr::between(all_others, min_valid_staff, max_valid_staff)]
          # get rid of nas. (Though wouldn't that be caught above? Hmph)
          all_others <- all_others[!is.na(all_others)]
          
          if (length ( all_others) > 0 ) {
            result[i] <- median ( all_others) 
          }

        }
      result
}

 # example:
 get_neighbors(c(.5, 1, 2, 5, 253, 7))

[1] 5.0 6.0 6.0 4.5 3.5 3.5

In [32]:

neighbors <- 
  staff_df |> 
  group_by ( facility_id ) |> 
  mutate ( neighbor_staff = get_neighbors( staff ) ) |> 
  ungroup() 





#### Create adjustment levels

Each value is checked against factors of 10 to see which one comes closest to its facility neighbors (all of them that were in some kind of possible range of 1 to 1500 employees). It picks the best value it can based on this. 

This fixes a lot of the truly 1/100th and 100x values. The problem comes in when it is clearly still a typo of some kind, or when there is a trend of (usually) vastly lower numbers of residents over the years. 

1. Check for outliers with 1 and not-1 multipliers. This will flag the staff numbers that are weird, but what we do with them is different. 
2. Check for reasonable levels of staff ratios. It's possible to have an outlier that comes from resident counts not from the adjustment or staff levels. 


The check at the end makes sure we don't pick an adjustment when none of the values are particularly close to one another. It only picks the adjustment of something other than 1 if that lands outside the 40% default tolerance level. 


In [33]:


my_factors <- c( 1/1000, 1/100,  1/10, 1, 10, 100, 1000 )


#@ value  = the original value
#@ target = the median value. 
#@ rel_tol = the tolerance should be within 40% of original value? 

best_factor_for <- function(value, target, factors = my_factors, rel_tol = .4) {
  if (is.na(target) || is.na(value)) return(NA_real_)
  # multiply the value by each factor
  candidates <- value * factors
  distances <- abs(candidates - target) 
  best_i <- which.min( distances) 
   # return the factor (.1, .01, 1, 10, 100, etc.) that results in the lowest difference between it and the median value.

  if (distances[best_i] <= rel_tol * target ) {
     factors[best_i]
  } else {
     1      
  }

}







Example of how this works , take the first row: 
: Staff is 0.15; staff_ratio is 0.0094; neighbor is 21. Others in this list range from .15-.29. 


id | Orig value | neighbor | d_1 | d_10 | d_100 | d10 < d1? | d100 < d10? | best_factor |
------ | --------| ---- | --- | --- | ---- | ---- | --- | --- |
Davis / Orman  | .15 | 21 | 20.85 | 19.5 | 6.0 | TRUE | TRUE |  100 |

It tends to work really well for values that are way too low, but is not sensitive enough to actual high numbers that are just weird. I need to set some parameters around when to use the adjustmant. 




In [34]:

tolerance <- 0.4  
# Set a really high max, but we'll check it again later anyway
min_max_ratio <- c( 0.2 , 6) 


staff_factors <- 
   neighbors |> 
   group_by ( facility_id ) |> 
   mutate ( 
      neighbor_ratio = get_neighbors( staff_ratio, min_valid = 0),
      best_factor = map2_dbl ( staff, neighbor_staff, \(x, y ) best_factor_for(x, y, my_factors) ) , 
      applied_factor = staff * best_factor, 
      ratio_corrected = applied_factor / residents ,

      orig_ratio_ok = between ( staff_ratio, min_max_ratio[1], min_max_ratio[2]) ,
      corr_ratio_ok = between ( ratio_corrected, min_max_ratio[1], min_max_ratio[2]) ,

      # this is checking for typical - this checks to see if it's way off of the median, in which case we don't like it.
      lands_near_typical = !is.na( neighbor_staff) & 
                           abs( applied_factor - neighbor_staff ) <= tolerance * neighbor_staff , 
      consistent_with_facility =  !is.na(neighbor_ratio) &
                           abs(staff_ratio - neighbor_ratio) <= tolerance * neighbor_ratio 
      
   )
    


Now check the factors and see whether we should keep the correction, keep the original, remove the value, or review. 


Here's a table of the decision. This assumes that there ARE good values for the neighbors, and the consistent ratios. That's not always true, so we have to deal with those conditions
I treat those that were changed differently than the ones that weren't changed -- more likely to keep it or review it rather than null it out.


Is the factor 1? |  Is the Fixed ratio OK? | Is fixed value consistent? | Is fixed staff near neighbor ? | Decision 
---| --- | --- | --- | --- | 
Y | Y | Anything | Anything  | Keep
Y | N | Y | Y | Review
Y | N | N | or  N | If either one is N, then null as outlier
N | Y | Y | Y | Fix
N | N | Y | Y | Review but fix in the meantime
N | Y | Y |  N | Fix
N | Y | N | Y | Null 
N | N | N | or N | ?? 

In [ ]:
decision_table <- 
  staff_factors |> 
  mutate ( is_1 = if_else ( is.na(best_factor) | best_factor == 1, TRUE, FALSE) ,
           ratio_corrected = coalesce (ratio_corrected, staff_ratio) ) |> 
  group_by ( is_1, corr_ratio_ok, lands_near_typical, consistent_with_facility) |> 
  summarize ( n = n() , .groups = "drop") |> 
  mutate ( verdict = case_when ( 
                  is_1 & corr_ratio_ok ~ "keep",
                  is_1 & (lands_near_typical | consistent_with_facility) ~ "review",
                  is_1  ~ "drop", 
                  corr_ratio_ok & (lands_near_typical | consistent_with_facility) ~ "fix",
                  corr_ratio_ok ~ "review",
                  !is_1 & !corr_ratio_ok ~ "drop"
           )
)
  

In [ ]:

get_verdict <- function ( is_1, ratio_ok, ratio_similar, staff_similar) {
  
  
  if ( is_1 ) {
    if ( ratio_ok ) {
      verdict <- "keep"
    } else {
      verdict <- "review"
    }

  } else {
    if (ratio_ok) {
        if ( ratio_similar | staff_similar) {
          verdict <- "fix"
        } else {
          verdict <- "remove"
        } 
    } else {
      
    }
  }
}


verdicts <- 
  staff_factors |> 
  # can we make a bit screen? 
  mutate ( bit_list = 
    
  )

  

  mutate ( verdict = 
                      ## Clearly OK
                      orig_ratio_ok & 


                  case_when ( 
                               orig_ratio_ok   ~ "keep",
                               corr_ratio_ok & best_factor != 1 & lands_near_typical ~ "fix",
                               corr_ratio_ok & best_factor != 1 & !lands_near_typical ~ "review", 
                               !orig_ratio_ok & !corr_ratio_ok ~ "remove", 
                               .default = "remove"
                  )
)